# KRUMP_CORE_V1 — Colab Smoke

ACE-Step Side-Step Corrected LoRAの成立確認だけを実行します。Full Training/Benchmarkは含めません。

In [ ]:
# 1. Ephemeral Colab-only paths: no persistent cloud storage
from pathlib import Path
import json, os, re, shutil, subprocess, sys, time, traceback, zipfile
WORK=Path('/content/KRUMP_CORE_V1_OUTPUT')
RUNTIME=Path('/content/KRUMP_CORE_V1_RUNTIME')
INPUT_ROOT=Path('/content/KRUMP_DATASET_V1')
WORK.mkdir(parents=True,exist_ok=True); RUNTIME.mkdir(parents=True,exist_ok=True); INPUT_ROOT.mkdir(parents=True,exist_ok=True)
os.environ.update({'UV_CACHE_DIR':str(RUNTIME/'uv-cache'),'PIP_NO_CACHE_DIR':'1','HF_HOME':str(RUNTIME/'hf-cache'),'HUGGINGFACE_HUB_CACHE':str(RUNTIME/'hf-cache'/'hub'),'MPLBACKEND':'Agg'})
LOG=WORK/'logs'/'smoke.log'; LOG.parent.mkdir(exist_ok=True)
def run(name,args,cwd=None,timeout=7200,input_text=None):
  p=subprocess.run(args,cwd=cwd,input=input_text,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout,check=False)
  LOG.open('a',encoding='utf-8').write(f'\n===== {name} =====\n{p.stdout}\n'); print(p.stdout[-6000:])
  if p.returncode: raise RuntimeError(f'{name} failed (exit={p.returncode}); see {LOG}')
  return p.stdout
def fail(stage,e):
  (WORK/'FAILED.json').write_text(json.dumps({'stage':stage,'error':str(e),'traceback':traceback.format_exc()},indent=2),encoding='utf-8'); raise e


In [ ]:
# 2. CUDA fail-fast: only T4 / L4 / A100 may train
try:
 import torch
 print(subprocess.check_output(['nvidia-smi','-L'],text=True)); print('torch=',torch.__version__)
 if not torch.cuda.is_available(): raise RuntimeError('CUDA unavailable')
 GPU=torch.cuda.get_device_name(0); print('GPU=',GPU); print('arch=',torch.cuda.get_arch_list())
 if not any(x in GPU for x in ('T4','L4','A100')): raise RuntimeError(f'Unsupported GPU: {GPU}; training not started')
except Exception as e: fail('gpu_preflight',e)


In [ ]:
# 3. Securely upload kaggle.json once, then download the existing Private Dataset to /content
try:
  KAGGLE_JSON=Path('/root/.kaggle/kaggle.json')
  if not KAGGLE_JSON.exists():
    from google.colab import files
    print('Upload only kaggle.json. It stays in this temporary Colab runtime and is never written to Drive or Notebook output.')
    uploaded=files.upload()
    if set(uploaded) != {'kaggle.json'}: raise RuntimeError('Upload exactly one file named kaggle.json.')
    KAGGLE_JSON.parent.mkdir(parents=True,exist_ok=True)
    KAGGLE_JSON.write_bytes(uploaded['kaggle.json'])
    os.chmod(KAGGLE_JSON,0o600)
  run('install Kaggle CLI',[sys.executable,'-m','pip','install','--quiet','--no-cache-dir','kaggle'],timeout=600)
  marker=INPUT_ROOT/'.dataset_ready'
  if not marker.exists():
    run('download Private Kaggle Dataset',['kaggle','datasets','download','-d','razorface/krump-training-v1-18-tracks','-p',str(INPUT_ROOT),'--unzip'],timeout=3600)
    nested=list(INPUT_ROOT.rglob('KRUMP_TRAINING_V1_18TRACKS.zip'))
    if nested:
      with zipfile.ZipFile(nested[0]) as z: z.extractall(INPUT_ROOT/'unpacked')
    marker.write_text('razorface/krump-training-v1-18-tracks',encoding='utf-8')
  print('KAGGLE_DATASET_DOWNLOAD_PASS')
except Exception as e: fail('kaggle_dataset_download',e)

# Validate the downloaded 18-track input. It remains under /content and is never modified.
try:
  files_meta=list(INPUT_ROOT.rglob('tracks.json'))
  if len(files_meta)!=1: raise RuntimeError(f'Expected one tracks.json, got {len(files_meta)}')
  tracks=json.loads(files_meta[0].read_text(encoding='utf-8'))
  if len(tracks)!=18: raise RuntimeError(f'Expected 18 tracks, got {len(tracks)}')
  audio={x.name:x for x in INPUT_ROOT.rglob('*') if x.is_file() and x.suffix.lower() in {'.mp3','.wav','.flac','.ogg','.m4a','.opus'}}
  missing=[x.get('file_name') for x in tracks if x.get('file_name') not in audio]
  if missing: raise RuntimeError(f'Missing audio: {missing[:3]}')
  print('DATASET_PASS: 18 tracks')
except Exception as e: fail('dataset_validation',e)


In [ ]:
# 4. Same ACE-Step 1.5 runtime and base model as Smoke Pipeline
try:
 ACE=RUNTIME/'ACE-Step-1.5'; CKPT=RUNTIME/'checkpoints'
 run('install uv',['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'],timeout=300)
 UV=shutil.which('uv') or '/root/.local/bin/uv'
 if not ACE.exists(): run('clone',['git','clone','https://github.com/ace-step/ACE-Step-1.5.git',str(ACE)],timeout=900)
 run('pin',['git','checkout','7202bc354d7fc31d1c0e5a90b0b49fb610e52362'],cwd=ACE,timeout=120)
 run('deps',[UV,'sync','--no-cache'],cwd=ACE,timeout=2400); CKPT.mkdir(exist_ok=True)
 link=ACE/'checkpoints'
 if link.exists() or link.is_symlink(): link.unlink() if (link.is_symlink() or link.is_file()) else shutil.rmtree(link)
 link.symlink_to(CKPT,target_is_directory=True)
 run('base model',[UV,'run','--no-sync','acestep-download','--model','acestep-v15-base','--dir',str(CKPT)],cwd=ACE,timeout=3600)
 if not any(CKPT.rglob('*.safetensors')): raise RuntimeError('Base weights missing')
 run('import',[UV,'run','python','-c','import acestep;print("ACE_STEP_IMPORT_PASS")'],cwd=ACE,timeout=300)
except Exception as e: fail('setup',e)


In [ ]:
# 5. Preprocess 18 tracks; tensors/manifest persist to Drive and are reused
try:
 STAGE=RUNTIME/'audio'; STAGE.mkdir(exist_ok=True); records=[]
 for row in tracks:
  src=audio[row['file_name']]; dst=STAGE/src.name
  if not dst.exists(): shutil.copy2(src,dst)
  r=dict(row); r.update(audio_path=str(dst),lyrics='[Instrumental]'); records.append(r)
 DATA=WORK/'dataset_krump_v1.json'; DATA.write_text(json.dumps(records,ensure_ascii=False,indent=2),encoding='utf-8')
 TENSORS=WORK/'preprocess'/'tensors'/'KRUMP_DATASET_V1'; MANIFEST=WORK/'preprocess'/'manifest.json'; TENSORS.mkdir(parents=True,exist_ok=True)
 if len(list(TENSORS.rglob('*.pt'))) < 18 or not MANIFEST.exists():
  run('preprocess',[UV,'run','python','-m','acestep.training_v2.cli.train_fixed','--checkpoint-dir',str(CKPT),'--model-variant','base','--preprocess','--dataset-json',str(DATA),'--audio-dir',str(STAGE),'--tensor-output',str(TENSORS),'--device','cuda:0','--precision','bf16'],cwd=ACE,timeout=10800)
  n=len(list(TENSORS.rglob('*.pt')));
  if n<18: raise RuntimeError(f'Preprocess {n}/18')
  MANIFEST.write_text(json.dumps({'status':'DONE','tracks':18,'tensors':str(TENSORS)},indent=2),encoding='utf-8')
 print('PREPROCESS_PASS',len(list(TENSORS.rglob('*.pt'))))
except Exception as e: fail('preprocess',e)


In [ ]:
# 6. Minimal Quick Side-Step Corrected LoRA: rank64 / alpha128 / batch1 / gradient checkpointing
try:
 QUICK=WORK/'quick_lora'; FINAL=QUICK/'final'; CKS=QUICK/'checkpoints'; QUICK.mkdir(parents=True,exist_ok=True)
 if not FINAL.exists() and not list(CKS.rglob('*')):
  cmd=[UV,'run','python','train.py','fixed','--checkpoint-dir',str(CKPT),'--model-variant','base','--dataset-dir',str(TENSORS),'--output-dir',str(QUICK),'--adapter-type','lora','--rank','64','--alpha','128','--dropout','0.15','--batch-size','1','--gradient-accumulation','4','--epochs','1','--save-every','1','--lr','5e-5','--device','cuda:0','--precision','bf16','--gradient-checkpointing','--offload-encoder']
  run('quick lora',cmd,cwd=ACE,timeout=7200,input_text='y\n')
 text=LOG.read_text(encoding='utf-8',errors='replace'); losses=re.findall(r'(?i)loss[^0-9]*([0-9]+(?:\.[0-9]+)?)',text)
 c=[str(x) for x in CKS.rglob('*')]; a=[str(x) for x in FINAL.rglob('*')] if FINAL.exists() else []
 report={'losses':losses[-10:],'checkpoint_path':str(CKS),'adapter_path':str(FINAL),'checkpoints':c[:50],'adapters':a[:50],'resume':bool(c or a)}
 (QUICK/'quick_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
 if not losses or not (c or a): raise RuntimeError('Quick LoRA has no numeric loss or checkpoint/adapter')
 print(json.dumps(report,indent=2))
except Exception as e: fail('quick_lora',e)


In [ ]:
# 7. Package and download the ephemeral Smoke artifacts before the Colab session ends
from google.colab import files
archive=Path('/content/KRUMP_CORE_V1_SMOKE_OUTPUT.zip')
if archive.exists(): archive.unlink()
shutil.make_archive(str(archive.with_suffix('')),'zip',WORK)
if not archive.exists() or archive.stat().st_size == 0: raise RuntimeError('Smoke output ZIP was not created.')
print({'zip':str(archive),'bytes':archive.stat().st_size})
files.download(str(archive))
